<a href="https://colab.research.google.com/github/anuvishalp/SQL-DataEngg-DB/blob/main/DiffDataAnalysisStyles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
####LIST OF ANALYTICS CONCEPTS (Instacart Case Study)
============================================================
Here are the 12 core concepts you should know:
============================================================
              Reorder Analysis
              Customer Churn / Retention
              Basket Size Analysis
              Basket Composition / Affinity
              Category Trend Analysis
              Department-Level Trend Analysis
              Aisle-Level Trend Analysis
              New vs Returning Customers
              Product Popularity Shift
              Stockout / Availability Proxy
              Seasonality Proxy
              Order Frequency Analysis

You will now get SQL for each concept, with Instacart‑specific examples.
================================================================================
🧩 1) Reorder Analysis -> Purpose : Measure how often customers reorder the same product → proxy for loyalty.

SQL: Compare Q2 vs Q3 reorder counts
sql
          SELECT
              p.product_id,
              prod.product_name,
              SUM(prior.reordered) AS prior_reorders,
              SUM(curr.reordered) AS current_reorders
          FROM ic_products prod
          LEFT JOIN ic_order_products_prior prior USING(product_id)
          LEFT JOIN ic_order_products_curr curr USING(product_id)
          GROUP BY p.product_id, prod.product_name
          ORDER BY current_reorders - prior_reorders DESC;
================================================================================
🧩 2) Customer Churn / Retention
Purpose
Identify customers who stopped ordering (churn) or continued (retained).

SQL

          WITH prior AS (
              SELECT DISTINCT user_id FROM ic_orders_prior
          ),
          curr AS (
              SELECT DISTINCT user_id FROM ic_orders_curr
          )
          SELECT
              (SELECT COUNT(*) FROM prior) AS q2_customers,
              (SELECT COUNT(*) FROM curr) AS q3_customers,
              (SELECT COUNT(*) FROM curr c LEFT JOIN prior p USING(user_id)
              WHERE p.user_id IS NULL) AS new_customers,
              (SELECT COUNT(*) FROM curr c JOIN prior p USING(user_id)) AS retained_customers;
================================================================================
🧩 3) Basket Size Analysis
Purpose
Understand how many items customers buy per order.

SQL
sql
          SELECT 'Q2' AS period, AVG(item_count) AS avg_basket_size
          FROM (
              SELECT order_id, COUNT(*) AS item_count
              FROM ic_order_products_prior
              GROUP BY order_id
          ) t

          UNION ALL

          SELECT 'Q3', AVG(item_count)
          FROM (
              SELECT order_id, COUNT(*) AS item_count
              FROM ic_order_products_curr
              GROUP BY order_id
          ) t;
================================================================================
🧩 4) Basket Composition / Affinity Analysis
Purpose
Find products frequently bought together.

SQL
          SELECT
              a.product_id AS product_a,
              b.product_id AS product_b,
              COUNT(*) AS times_bought_together
          FROM ic_order_products_curr a
          JOIN ic_order_products_curr b
              ON a.order_id = b.order_id
            AND a.product_id < b.product_id
          GROUP BY a.product_id, b.product_id
          ORDER BY times_bought_together DESC;
================================================================================
🧩 5) Category Trend Analysis (Department-Level)
Purpose
See which departments grew or shrank.

SQL
            WITH prior AS (
                SELECT d.department, COUNT(*) AS cnt
                FROM ic_order_products_prior p
                JOIN ic_products prod USING(product_id)
                JOIN ic_departments d USING(department_id)
                GROUP BY d.department
            ),
            curr AS (
                SELECT d.department, COUNT(*) AS cnt
                FROM ic_order_products_curr c
                JOIN ic_products prod USING(product_id)
                JOIN ic_departments d USING(department_id)
                GROUP BY d.department
            )
            SELECT
                COALESCE(prior.department, curr.department) AS department,
                prior.cnt AS q2_count,
                curr.cnt AS q3_count,
                curr.cnt - prior.cnt AS change_in_items
            FROM prior
            FULL JOIN curr USING(department);
================================================================================
🧩 6) Aisle-Level Trend Analysis
Purpose
Granular version of category trends.

SQL

          WITH prior AS (
              SELECT a.aisle, COUNT(*) AS cnt
              FROM ic_order_products_prior p
              JOIN ic_products prod USING(product_id)
              JOIN ic_aisles a USING(aisle_id)
              GROUP BY a.aisle
          ),
          curr AS (
              SELECT a.aisle, COUNT(*) AS cnt
              FROM ic_order_products_curr c
              JOIN ic_products prod USING(product_id)
              JOIN ic_aisles a USING(aisle_id)
              GROUP BY a.aisle
          )
          SELECT
              COALESCE(prior.aisle, curr.aisle) AS aisle,
              prior.cnt AS q2_count,
              curr.cnt AS q3_count,
              curr.cnt - prior.cnt AS change_in_items
          FROM prior
          FULL JOIN curr USING(aisle);
================================================================================
🧩 7) New vs Returning Customers (Behavioral Shift)
Purpose
Understand customer base changes.

SQL

            SELECT
                CASE
                    WHEN p.user_id IS NULL THEN 'New in Q3'
                    ELSE 'Returning'
                END AS customer_type,
                COUNT(*) AS num_customers
            FROM ic_orders_curr c
            LEFT JOIN ic_orders_prior p USING(user_id)
            GROUP BY customer_type;
================================================================================
🧩 8) Product Popularity Shift
Purpose
Find products that gained or lost popularity.

SQL
sql
WITH prior AS (
    SELECT product_id, COUNT(*) AS cnt
    FROM ic_order_products_prior
    GROUP BY product_id
),
curr AS (
    SELECT product_id, COUNT(*) AS cnt
    FROM ic_order_products_curr
    GROUP BY product_id
)
SELECT
    COALESCE(prior.product_id, curr.product_id) AS product_id,
    prior.cnt AS q2_count,
    curr.cnt AS q3_count,
    curr.cnt - prior.cnt AS change_in_orders
FROM prior
FULL JOIN curr USING(product_id)
ORDER BY change_in_orders DESC;
🧩 9) Availability Proxy (Stockout Detection)
Purpose
If a product appears in Q3 but not Q2 → likely newly available.

SQL
sql
SELECT
    curr.product_id,
    COUNT(curr.product_id) AS q3_orders,
    COALESCE(prior.product_id, NULL) AS existed_in_q2
FROM ic_order_products_curr curr
LEFT JOIN ic_order_products_prior prior USING(product_id)
WHERE prior.product_id IS NULL
GROUP BY curr.product_id;
🧩 10) Seasonality Proxy
Purpose
Detect seasonal shifts without dates.

SQL
sql
SELECT
    d.department,
    SUM(curr.reordered) - SUM(prior.reordered) AS reorder_change
FROM ic_products prod
JOIN ic_departments d USING(department_id)
LEFT JOIN ic_order_products_prior prior USING(product_id)
LEFT JOIN ic_order_products_curr curr USING(product_id)
GROUP BY d.department
ORDER BY reorder_change DESC;
🧩 11) Order Frequency Analysis
Purpose
How often customers order.

SQL
sql
SELECT user_id, COUNT(*) AS num_orders
FROM ic_orders_curr
GROUP BY user_id
ORDER BY num_orders DESC;
🧩 12) High-Level Trend Summary (All Concepts Combined)
Purpose
One unified snapshot.

SQL
sql
SELECT
    'reorders' AS metric,
    SUM(curr.reordered) - SUM(prior.reordered) AS change_value
FROM ic_order_products_prior prior
JOIN ic_order_products_curr curr USING(product_id)

UNION ALL

SELECT
    'total_items',
    (SELECT COUNT(*) FROM ic_order_products_curr) -
    (SELECT COUNT(*) FROM ic_order_products_prior)

UNION ALL

SELECT
    'unique_products',
    (SELECT COUNT(DISTINCT product_id) FROM ic_order_products_curr) -
    (SELECT COUNT(DISTINCT product_id) FROM ic_order_products_prior);
🧾 FINAL CHEATSHEET — Analysis Types & SQL Concepts
Analysis Type	What It Measures	SQL Concept	Guided Link
Reorder Analysis	Loyalty, repeat demand	SUM(reordered)	reorder analysis
Customer Churn	Who left vs stayed	DISTINCT user_id	customer churn
Basket Size	Items per order	COUNT(*) per order	basket analysis
Basket Composition	Items bought together	Self-join on order_id	basket composition
Category Trends	Dept-level demand	GROUP BY department	category trends
Aisle Trends	Aisle-level demand	GROUP BY aisle	aisle trends
Product Popularity	Winners & losers	COUNT(product_id)	popularity shift
New vs Returning	Acquisition vs retention	LEFT JOIN customers	new vs returning
Availability Proxy	Stockouts / new items	Missing in prior	availability proxy
Seasonality Proxy	Seasonal demand	Compare Q2 vs Q3	seasonality proxy
Order Frequency	Customer activity	COUNT orders	order frequency



In [ ]:
1️⃣ Reorder Analysis
What it is
Reorder analysis measures how often customers buy the same product again.
It’s a proxy for loyalty, satisfaction, availability, and habit formation.

Why it matters
If reorder rates increase:

Customers trust the product

The item is consistently available

The season or trend favors it

If reorder rates drop:

Quality issues

Stockouts

Competitor alternatives

Price changes

Instacart Example
Comparing Q2 vs Q3 reorder counts for “Organic Navel Oranges” shows a jump from 0 → 30.
This signals a behavioral shift worth investigating.

SQL Signal
Count reordered = 1 per product in prior vs current tables.

Guided Link
Explore deeper: reorder analysis

2️⃣ Customer Churn
What it is
Churn measures how many customers stop buying from a platform over time.

Why it matters
High churn =

Poor experience

High competition

Low loyalty

Weak retention strategy

Low churn =

Strong product-market fit

Good customer experience

Effective loyalty programs

Instacart Example
If many customers appear in Q2 but not in Q3 → churn is high.
If many customers appear in both → retention is strong.

SQL Signal
Compare distinct customers in Q2 vs Q3.

Guided Link
Explore deeper: customer churn

3️⃣ Basket Analysis
What it is
Basket analysis studies what customers buy together and how large their orders are.

It includes:

Basket size (number of items per order)

Basket value (total spend per order)

Product affinity (items frequently bought together)

Why it matters
It reveals:

Customer intent (stock-up vs quick trip)

Seasonal patterns

Cross-sell opportunities

Promotion effectiveness

Instacart Example
If Q3 baskets contain more produce items, it may indicate:

Summer season

Healthy eating trends

Produce promotions

SQL Signal
Count items per order → compare Q2 vs Q3.

Guided Link
Explore deeper: basket analysis

4️⃣ Category Trends
What it is
Category trend analysis measures how demand changes across departments or aisles.

Examples:

Produce

Snacks

Dairy

Household goods

Why it matters
It helps answer:

Which categories are growing?

Which are declining?

Are trends seasonal or structural?

Should Instacart adjust inventory or promotions?

Instacart Example
If produce orders increase sharply in Q3:

Seasonal fruits

Summer recipes

Health-conscious behavior

SQL Signal
Count items per department in Q2 vs Q3 → compare.

Guided Link
Explore deeper: category trends

🧩 Summary Table (Copy‑Paste Ready)
Concept	Simple Definition	What It Reveals	Instacart Example	Guided Link
Reorder Analysis	How often customers rebuy items	Loyalty, satisfaction	Oranges reordered 0→30	reorder analysis
Customer Churn	How many customers stop buying	Retention health	Q2 customers missing in Q3	customer churn
Basket Analysis	What customers buy & how much	Behavior, cross-sell	Larger produce baskets in Q3	basket analysis
Category Trends	Which departments grow/decline	Demand shifts	Produce ↑, snacks ↓	category trends

